<a href="https://colab.research.google.com/github/stekkos89thelawnmower/Flamingo-Qd-Analysis/blob/main/FLAMINGO_QD_highpower_specificity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FLAMINGO Q_D — High-Power Verification and Specificity Check (8-Independent-Seed Method)

Self-contained pipeline for the deepest robustness check of this project: replacing the single-seed jackknife estimate with a one-sample t-test across 8 independent seed realisations (N=6000 domains at R=15 Mpc, N=3000 at R=30 Mpc — near the geometric packing limit for each radius), applied uniformly to all 5 mass bins and all three comparison variants (NoCooling, Jet, fgas-8sigma).

**Two findings, in opposite directions:**
1. The transition zone (bins 1-3), previously judged unresolved under the standard N=800/500 protocol, shows a significant NoCooling signal under this more powerful method — but the *same* signal, of equal or greater amplitude, appears for the Jet and fgas-8sigma control variants. This rules the transition-zone signal out as specific to radiative cooling (Report Section 6.6).
2. The same method applied to the two already-established pillars (bins 0 and 4) shows the *opposite* pattern: Jet and fgas-8sigma have a real, statistically robust effect there too, but of systematically *opposite sign* to NoCooling — the most direct evidence available that the pillars reflect specific, distinguishable cooling physics, strengthened rather than weakened by this check.

Cells are organised bin by bin (0 through 4), each variant/radius combination in its own cell, matching the crash-resistant workflow used to originally collect this data (long-running multi-variant cells proved unreliable on the interactive runtime used for this analysis).

## 0. Setup

In [ ]:
!pip install hdfstream -q
import numpy as np
import hdfstream
import time
from scipy.spatial import cKDTree
from scipy.stats import wilcoxon, ttest_1samp


In [ ]:
GRAMS_PER_MSUN = 1.988409870698051e33
CM_PER_KM = 1e5
MASS_CUT = 1e12
N_GRID = 200
K_NEIGHBORS = 16
BOX_SIDE = 1000.0
CELL_SIZE = BOX_SIDE / N_GRID
GRID_1D = np.linspace(0, BOX_SIDE, N_GRID, endpoint=False) + CELL_SIZE / 2
SEED = 42
SNAPSHOT_Z0 = "halo_properties_0077.hdf5"  # z=0.0, L1_m9 snapshot numbering
MASS_LO_EDGE = 1.21e12
MASS_HI_EDGE = 5.91e12
N_TRANSITION_BINS = 3
TARGET_N = 1348496  # min tracer count across fiducial/NoCooling/Jet/fgas-8sigma at z=0.0

VARIANT_A = {"run_path": "L1_m9/L1_m9", "label": "fiducial"}
VARIANT_B = {"run_path": "L1_m9/NoCooling", "label": "NoCooling"}
VARIANT_JET = {"run_path": "L1_m9/Jet", "label": "Jet"}
VARIANT_FGAS8SIGMA = {"run_path": "L1_m9/fgas-8sigma", "label": "fgas-8sigma"}

REPLICATION_SEEDS_EXTENDED = [42, 123, 456, 789, 111, 222, 333, 999]


## 1. Core pipeline functions

In [ ]:
def load_tracers_velocities_masses(flamingo_dir, run_path, snapshot_file, box_side=BOX_SIDE, expected_z=None):
    soap_file = flamingo_dir[f"{run_path}/SOAP-HBT/{snapshot_file}"]
    z = float(np.array(soap_file["Header"].attrs["Redshift"]).squeeze())
    if expected_z is not None:
        assert abs(z - expected_z) < 1e-6, f"z={z}, expected {expected_z}"
    total_mass_raw = np.array(soap_file["SO/200_crit/TotalMass"][:], dtype=np.float64)
    conv_mass = dict(soap_file["SO/200_crit/TotalMass"].attrs)
    cgs_mass = float(np.array(conv_mass["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    mass_msun = total_mass_raw * cgs_mass / GRAMS_PER_MSUN
    mask = mass_msun > MASS_CUT
    pos_raw = np.mod(np.array(soap_file["SO/200_crit/CentreOfMass"][:], dtype=np.float32), box_side)
    vel_raw = np.array(soap_file["SO/200_crit/CentreOfMassVelocity"][:], dtype=np.float32)
    conv_vel = dict(soap_file["SO/200_crit/CentreOfMassVelocity"].attrs)
    vel_cgs_factor = float(np.array(conv_vel["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    vel_kms = vel_raw * vel_cgs_factor / CM_PER_KM
    return pos_raw[mask], vel_kms[mask], mass_msun[mask].astype(np.float32), z


def periodic_delta(a, b, box_side):
    d = np.abs(a - b)
    return np.minimum(d, box_side - d)


def make_observer_positions(n_observers, min_separation_mpc, box_side, seed=SEED, max_tries_factor=500):
    rng = np.random.default_rng(seed)
    positions = []
    tries = 0
    max_tries = max_tries_factor * n_observers
    while len(positions) < n_observers and tries < max_tries:
        cand = rng.uniform(0, box_side, size=3)
        if all(np.sqrt((periodic_delta(cand, p, box_side) ** 2).sum()) >= min_separation_mpc for p in positions):
            positions.append(cand)
        tries += 1
    return np.array(positions)


def build_grid_tree(grid_1d, box_side):
    gx, gy, gz = np.meshgrid(grid_1d, grid_1d, grid_1d, indexing="ij")
    grid_points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    return cKDTree(grid_points, boxsize=box_side)


def subsample_to_match_with_mass(tracers, velocities, masses, target_n, seed):
    n = len(tracers)
    if n <= target_n:
        return tracers, velocities, masses
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=target_n, replace=False)
    return tracers[idx], velocities[idx], masses[idx]


def build_velocity_field(tracers, velocities, grid_1d, box_side, n_grid, k_neighbors):
    tree = cKDTree(tracers, boxsize=box_side)
    vx = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    vy = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    vz = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    for i in range(n_grid):
        gx = np.full(n_grid * n_grid, grid_1d[i], dtype=np.float32)
        gyv, gzv = np.meshgrid(grid_1d, grid_1d, indexing="ij")
        pts = np.stack([gx, gyv.ravel().astype(np.float32), gzv.ravel().astype(np.float32)], axis=1)
        d, idx = tree.query(pts, k=k_neighbors)
        d = np.maximum(d, 1e-6)
        w = 1.0 / d**2
        w /= w.sum(axis=1, keepdims=True)
        vx[i] = (w * velocities[idx, 0]).sum(axis=1).reshape(n_grid, n_grid)
        vy[i] = (w * velocities[idx, 1]).sum(axis=1).reshape(n_grid, n_grid)
        vz[i] = (w * velocities[idx, 2]).sum(axis=1).reshape(n_grid, n_grid)
    return vx, vy, vz


def compute_theta_sigma2(vx, vy, vz, cell_size):
    dvx_dx = (np.roll(vx,-1,0)-np.roll(vx,1,0))/(2*cell_size)
    dvx_dy = (np.roll(vx,-1,1)-np.roll(vx,1,1))/(2*cell_size)
    dvx_dz = (np.roll(vx,-1,2)-np.roll(vx,1,2))/(2*cell_size)
    dvy_dx = (np.roll(vy,-1,0)-np.roll(vy,1,0))/(2*cell_size)
    dvy_dy = (np.roll(vy,-1,1)-np.roll(vy,1,1))/(2*cell_size)
    dvy_dz = (np.roll(vy,-1,2)-np.roll(vy,1,2))/(2*cell_size)
    dvz_dx = (np.roll(vz,-1,0)-np.roll(vz,1,0))/(2*cell_size)
    dvz_dy = (np.roll(vz,-1,1)-np.roll(vz,1,1))/(2*cell_size)
    dvz_dz = (np.roll(vz,-1,2)-np.roll(vz,1,2))/(2*cell_size)
    theta = dvx_dx+dvy_dy+dvz_dz
    grad = np.array([[dvx_dx,dvx_dy,dvx_dz],[dvy_dx,dvy_dy,dvy_dz],[dvz_dx,dvz_dy,dvz_dz]])
    sym = 0.5*(grad+grad.transpose(1,0,2,3,4))
    trace_third = theta/3.0
    sigma = sym.copy()
    for i in range(3):
        sigma[i,i] -= trace_third
    sigma2 = np.sum(sigma**2, axis=(0,1))
    return theta, sigma2


def bootstrap_ci(delta, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(delta)
    boot_means = np.array([rng.choice(delta, size=n, replace=True).mean() for _ in range(n_boot)])
    return np.percentile(boot_means, [2.5, 97.5])


def QD_per_domain(theta, sigma2, grid_tree, observer_positions, radius_mpc):
    theta_flat = theta.ravel(); sigma2_flat = sigma2.ravel()
    qd_values = np.empty(len(observer_positions))
    for i, obs in enumerate(observer_positions):
        idx = grid_tree.query_ball_point(obs, r=radius_mpc)
        th = theta_flat[idx]; s2 = sigma2_flat[idx]
        var_theta = (th**2).mean() - th.mean()**2
        qd_values[i] = (2.0/3.0)*var_theta - s2.mean()
    return qd_values


def paired_QD_comparison(theta_A, sigma2_A, theta_B, sigma2_B, grid_tree,
                          observer_positions, radius_mpc, label_A="A", label_B="B", verbose=True):
    qd_A = QD_per_domain(theta_A, sigma2_A, grid_tree, observer_positions, radius_mpc)
    qd_B = QD_per_domain(theta_B, sigma2_B, grid_tree, observer_positions, radius_mpc)
    delta = qd_B - qd_A
    ci_lo, ci_hi = bootstrap_ci(delta)
    try:
        _, p_wilcoxon = wilcoxon(qd_B, qd_A)
    except ValueError:
        p_wilcoxon = float("nan")
    if verbose:
        print(f"R={radius_mpc:.0f} Mpc, N={len(observer_positions)}: "
              f"{label_A}={qd_A.mean():.3f}  {label_B}={qd_B.mean():.3f}  "
              f"Delta={delta.mean():.3f}  CI95%=[{ci_lo:.3f},{ci_hi:.3f}]  p_Wilcoxon={p_wilcoxon:.2e}")
    return {"qd_A": qd_A, "qd_B": qd_B, "delta": delta, "ci_lo": ci_lo, "ci_hi": ci_hi, "p_wilcoxon": p_wilcoxon}


def build_fields_refined_bins(tracers, velocities, masses, mass_lo_edge, mass_hi_edge, n_transition_bins):
    edges_all = [masses.min(), mass_lo_edge]
    mid_mask = (masses >= mass_lo_edge) & (masses < mass_hi_edge)
    mid_edges = np.quantile(masses[mid_mask], np.linspace(0, 1, n_transition_bins + 1))
    edges_all += list(mid_edges[1:])
    edges_all += [masses.max()]
    edges_all = np.array(edges_all)
    out = []
    for i in range(len(edges_all) - 1):
        lo, hi = edges_all[i], edges_all[i + 1]
        sel = (masses >= lo) & (masses <= hi if i == len(edges_all) - 2 else masses < hi)
        if sel.sum() < 500:
            out.append({"theta": None, "sigma2": None, "N": int(sel.sum()), "mass_median": np.nan})
            continue
        vx, vy, vz = build_velocity_field(tracers[sel], velocities[sel], GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
        theta, sigma2 = compute_theta_sigma2(vx.astype(np.float64), vy.astype(np.float64), vz.astype(np.float64), CELL_SIZE)
        out.append({"theta": theta, "sigma2": sigma2, "N": int(sel.sum()), "mass_median": float(np.median(masses[sel]))})
    return edges_all, out


def run_extended_test_variant(fields_a, fields_b, bin_idx, R, N, seeds, label_a="fiducial", label_b="variant"):
    """Delta = variant - fiducial. Each of the 8 seeds is treated as an
    independent measurement of the mean Delta; significance comes from a
    one-sample t-test across the 8 seed-level means, NOT from any single
    seed's internal (noisier, 8-octant) jackknife estimate."""
    print(f"\n{label_b} vs {label_a}, Bin {bin_idx}, R={R:.0f} Mpc, N={N}, {len(seeds)} seeds")
    delta_list = []
    f_a = fields_a[bin_idx]
    f_b = fields_b[bin_idx]
    for s in seeds:
        obs_pos = make_observer_positions(N, 2 * R, BOX_SIDE, seed=s)
        res = paired_QD_comparison(f_a["theta"], f_a["sigma2"], f_b["theta"], f_b["sigma2"],
                                    grid_tree, obs_pos, R, label_a, label_b, verbose=False)
        delta_list.append(res["delta"].mean())

    delta_arr = np.array(delta_list)
    t_stat, p_val = ttest_1samp(delta_arr, 0)
    n_pos = int((delta_arr > 0).sum())
    print(f"  mean={delta_arr.mean():+.4f}  t={t_stat:+.2f}  p={p_val:.4f}  positive={n_pos}/{len(seeds)}")
    return {"delta_list": delta_list, "t_stat": t_stat, "p_val": p_val, "n_pos": n_pos}


def benjamini_hochberg(tests, alpha=0.05):
    sorted_tests = sorted(tests, key=lambda x: x[1])
    m = len(sorted_tests)
    max_i = 0
    for i, tt in enumerate(sorted_tests, start=1):
        if tt[1] <= (i / m) * alpha:
            max_i = i
    return [(name, pval, i <= max_i) for i, (name, pval) in enumerate(sorted_tests, start=1)]


## 2. Download and build fields (all 4 variants, all 5 mass bins)

This is the expensive step. Everything after this reuses these fields and runs quickly. If the runtime disconnects partway through the individual test cells below, only this cell needs to be rerun -- results already printed from completed cells are not lost as long as you keep a copy of the printed output.

In [ ]:
root_dir = hdfstream.open("cosma", "/")
flamingo_dir = root_dir["FLAMINGO"]

print("Downloading fiducial, z=0.0...")
tr_f, vel_f, mass_f, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_A["run_path"], SNAPSHOT_Z0, expected_z=0.0)
print(f"  {len(tr_f)} tracers")
print("Downloading NoCooling, z=0.0...")
tr_n, vel_n, mass_n, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_B["run_path"], SNAPSHOT_Z0, expected_z=0.0)
print(f"  {len(tr_n)} tracers")
print("Downloading Jet, z=0.0...")
tr_jet, vel_jet, mass_jet, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_JET["run_path"], SNAPSHOT_Z0, expected_z=0.0)
print(f"  {len(tr_jet)} tracers")
print("Downloading fgas-8sigma, z=0.0...")
tr_fgas, vel_fgas, mass_fgas, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_FGAS8SIGMA["run_path"], SNAPSHOT_Z0, expected_z=0.0)
print(f"  {len(tr_fgas)} tracers")

print(f"\nN-matching to {TARGET_N} tracers")

t0 = time.time()
tr_f_m, vel_f_m, mass_f_m = subsample_to_match_with_mass(tr_f, vel_f, mass_f, TARGET_N, seed=SEED)
edges_refined, fields_fiducial = build_fields_refined_bins(tr_f_m, vel_f_m, mass_f_m, MASS_LO_EDGE, MASS_HI_EDGE, N_TRANSITION_BINS)
print(f"  fiducial fields built ({time.time()-t0:.0f}s)")

tr_n_m, vel_n_m, mass_n_m = subsample_to_match_with_mass(tr_n, vel_n, mass_n, TARGET_N, seed=SEED)
_, fields_nocooling = build_fields_refined_bins(tr_n_m, vel_n_m, mass_n_m, MASS_LO_EDGE, MASS_HI_EDGE, N_TRANSITION_BINS)
print(f"  NoCooling fields built ({time.time()-t0:.0f}s)")

tr_jet_m, vel_jet_m, mass_jet_m = subsample_to_match_with_mass(tr_jet, vel_jet, mass_jet, TARGET_N, seed=SEED)
_, fields_jet = build_fields_refined_bins(tr_jet_m, vel_jet_m, mass_jet_m, MASS_LO_EDGE, MASS_HI_EDGE, N_TRANSITION_BINS)
print(f"  Jet fields built ({time.time()-t0:.0f}s)")

tr_fgas_m, vel_fgas_m, mass_fgas_m = subsample_to_match_with_mass(tr_fgas, vel_fgas, mass_fgas, TARGET_N, seed=SEED)
_, fields_fgas = build_fields_refined_bins(tr_fgas_m, vel_fgas_m, mass_fgas_m, MASS_LO_EDGE, MASS_HI_EDGE, N_TRANSITION_BINS)
print(f"  fgas-8sigma fields built ({time.time()-t0:.0f}s)")

print(f"\nBin edges (Msun): {[f'{e:.2e}' for e in edges_refined]}")

grid_tree = build_grid_tree(GRID_1D, BOX_SIDE)

results = {}  # collects every test result, keyed "Variant_binN_R##"
print("\nSetup complete. Proceed to the individual bin cells below.")


## 3. Bin 0 (lowest mass, ~1.10e12 Msun) -- established low-mass pillar

NoCooling is the original, positive, already-well-established signal here. Jet and fgas-8sigma are the specificity check: do they show anything, and if so, in which direction?

In [ ]:
results["NoCooling_bin0_R15"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 0, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["NoCooling_bin0_R30"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 0, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["Jet_bin0_R15"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 0, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["Jet_bin0_R30"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 0, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["fgas8sigma_bin0_R15"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 0, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


In [ ]:
results["fgas8sigma_bin0_R30"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 0, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


## 4. Bin 1 (~1.40e12 Msun) -- transition zone

In [ ]:
results["NoCooling_bin1_R15"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 1, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["NoCooling_bin1_R30"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 1, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["Jet_bin1_R15"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 1, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["Jet_bin1_R30"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 1, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["fgas8sigma_bin1_R15"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 1, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


In [ ]:
results["fgas8sigma_bin1_R30"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 1, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


## 5. Bin 2 (~2.00e12 Msun) -- transition zone

In [ ]:
results["NoCooling_bin2_R15"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 2, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["NoCooling_bin2_R30"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 2, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["Jet_bin2_R15"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 2, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["Jet_bin2_R30"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 2, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["fgas8sigma_bin2_R15"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 2, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


In [ ]:
results["fgas8sigma_bin2_R30"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 2, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


## 6. Bin 3 (~3.56e12 Msun) -- transition zone

In [ ]:
results["NoCooling_bin3_R15"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 3, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["NoCooling_bin3_R30"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 3, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["Jet_bin3_R15"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 3, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["Jet_bin3_R30"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 3, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["fgas8sigma_bin3_R15"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 3, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


In [ ]:
results["fgas8sigma_bin3_R30"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 3, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


## 7. Bin 4 (highest mass, ~1.18e13 Msun) -- established high-mass pillar

NoCooling is the original, strongly negative, already-well-established signal here. Jet and fgas-8sigma are the specificity check.

In [ ]:
results["NoCooling_bin4_R15"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 4, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["NoCooling_bin4_R30"] = run_extended_test_variant(
    fields_fiducial, fields_nocooling, 4, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "NoCooling")


In [ ]:
results["Jet_bin4_R15"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 4, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["Jet_bin4_R30"] = run_extended_test_variant(
    fields_fiducial, fields_jet, 4, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "Jet")


In [ ]:
results["fgas8sigma_bin4_R15"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 4, 15.0, 6000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


In [ ]:
results["fgas8sigma_bin4_R30"] = run_extended_test_variant(
    fields_fiducial, fields_fgas, 4, 30.0, 3000, REPLICATION_SEEDS_EXTENDED, "fiducial", "fgas-8sigma")


## 8. Final summary: specificity by bin, Benjamini-Hochberg correction, comparison plot

Run this only after all 30 cells above have completed (5 bins x 3 variants x 2 radii).

In [ ]:
import matplotlib.pyplot as plt

print(f"Collected {len(results)}/30 tests\n")

print(f"{'Case':<22s} {'t':>8s} {'p':>10s}")
for key, r in results.items():
    print(f"{key:<22s} {r['t_stat']:8.2f} {r['p_val']:10.4f}")


In [ ]:
# --- Benjamini-Hochberg on the 24 control tests (Jet + fgas-8sigma across all 5 bins) ---
control_tests = [(k, v["p_val"]) for k, v in results.items() if not k.startswith("NoCooling")]
print(f"Benjamini-Hochberg correction (alpha=0.05), {len(control_tests)} control tests (Jet + fgas-8sigma):\n")
n_survive = 0
for name, p, survives in benjamini_hochberg(control_tests):
    marker = "  <-- SURVIVES" if survives else ""
    if survives:
        n_survive += 1
    print(f"  {name:<22s} p={p:.2e}{marker}")
print(f"\nSurvive: {n_survive}/{len(control_tests)}")


In [ ]:
# --- side-by-side comparison table, t-statistic, all 5 bins x 2 radii ---
print(f"{'Bin/radius':<12s} {'t NoCooling':>12s} {'t Jet':>10s} {'t fgas-8sigma':>14s}")
for bin_idx in range(5):
    for R in [15, 30]:
        key_nc = f"NoCooling_bin{bin_idx}_R{R}"
        key_jet = f"Jet_bin{bin_idx}_R{R}"
        key_fgas = f"fgas8sigma_bin{bin_idx}_R{R}"
        t_nc = results[key_nc]["t_stat"] if key_nc in results else float("nan")
        t_jet = results[key_jet]["t_stat"] if key_jet in results else float("nan")
        t_fgas = results[key_fgas]["t_stat"] if key_fgas in results else float("nan")
        print(f"Bin{bin_idx} R={R:<6d} {t_nc:12.2f} {t_jet:10.2f} {t_fgas:14.2f}")


In [ ]:
# --- comparison plot: t-statistic vs bin, one panel per radius, three lines ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
colors = {"NoCooling": "tab:blue", "Jet": "tab:green", "fgas8sigma": "tab:red"}
labels_display = {"NoCooling": "NoCooling", "Jet": "Jet", "fgas8sigma": "fgas-8sigma"}

for ax, R in zip(axes, [15, 30]):
    for prefix in ["NoCooling", "Jet", "fgas8sigma"]:
        t_vals = []
        bins_present = []
        for bin_idx in range(5):
            key = f"{prefix}_bin{bin_idx}_R{R}"
            if key in results:
                t_vals.append(results[key]["t_stat"])
                bins_present.append(bin_idx)
        ax.plot(bins_present, t_vals, "o-", color=colors[prefix], label=labels_display[prefix])
    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.axhline(2, color="lightgrey", linestyle=":", linewidth=1)
    ax.axhline(-2, color="lightgrey", linestyle=":", linewidth=1)
    ax.set_xlabel("Mass bin")
    ax.set_xticks(range(5))
    ax.set_title(f"R = {R} Mpc")
    ax.legend()
axes[0].set_ylabel("t-statistic (8-seed one-sample t-test)")
plt.suptitle("Sign convergence in the transition zone (bins 1-3) vs sign divergence at the pillars (bins 0, 4)")
plt.tight_layout()
plt.savefig("highpower_specificity_all_bins.png", dpi=150)
plt.show()
